# 07. Microcase: A/B Testing (Email Campaigns)

This is an independent microcase, using the Hillstrom MineThatData dataset, not connected to dunnhumby or PostgreSQL. It answers: did either of the
two email campaigns (Mens, Womens) produce a statistically significant
effect on conversion or spend, compared to the control group (No E-Mail)?

This yields 4 hypothesis tests (2 treatment groups × 2 metrics), which is
exactly the scenario where multiple-comparisons correction is warranted, 
unlike earlier notebooks in this project, where it was deliberately avoided
for lacking justification. **Bonferroni** is used here rather than
Benjamini-Hochberg (used in a prior project): with only 4 tests,
Benjamini-Hochberg's main advantage, retaining statistical power across
many comparisons, barely applies, so there's no real benefit to justify
the more complex method here.

Two different statistical tests are used, matched to what each metric
actually is:
- **Conversion** (binary: did the customer make a purchase or not) is
  tested with a two-proportion z-test.
- **Spend** (continuous dollar amount, zero for non-converters) is tested
  with a two-sample t-test.

## 1. Setup and Group Sizes

Before testing anything, we inspect the file itself, the same standard applied to every dataset in this project: columns, dtypes, nulls, and a sample of rows, rather than assuming `segment`/`conversion`/`spend` are the only columns or that they hold clean values.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests

RAW_DATA_DIR = Path("../data/raw/Kevin Hillstrom MineThatData E-MailAnalytics")
filepath = RAW_DATA_DIR / "hillstrom.csv"

df = pd.read_csv(filepath)

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print("\nDtypes:")
print(df.dtypes)
print("\nNulls per column:")
print(df.isna().sum())
df.head()

Rows: 64000
Columns: ['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion', 'spend']

Dtypes:
recency              int64
history_segment     object
history            float64
mens                 int64
womens               int64
zip_code            object
newbie               int64
channel             object
segment             object
visit                int64
conversion           int64
spend              float64
dtype: object

Nulls per column:
recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
visit              0
conversion         0
spend              0
dtype: int64


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


In [2]:
print(f"conversion values: {sorted(df['conversion'].unique())}")
print(f"Rows with spend < 0: {(df['spend'] < 0).sum()}")
print(f"Rows with conversion == 1 but spend == 0: {((df['conversion'] == 1) & (df['spend'] == 0)).sum()}")
print(f"Rows with conversion == 0 but spend > 0: {((df['conversion'] == 0) & (df['spend'] > 0)).sum()}")

conversion values: [np.int64(0), np.int64(1)]
Rows with spend < 0: 0
Rows with conversion == 1 but spend == 0: 0
Rows with conversion == 0 but spend > 0: 0


`conversion` is confirmed strictly binary, `spend` has no negative values, and `conversion`/`spend` agree with each other (nobody converts with zero spend or spends without converting), so the two metrics tested below are measuring consistent things rather than two independently-computed columns that happen to disagree. With that confirmed, we look at the three experimental groups themselves.

In [3]:
print("Group sizes:")
print(df["segment"].value_counts())

print("\nConversion rate and mean spend by group:")
print(df.groupby("segment")[["conversion", "spend"]].mean())

Group sizes:
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

Conversion rate and mean spend by group:
               conversion     spend
segment                            
Mens E-Mail      0.012531  1.422617
No E-Mail        0.005726  0.652789
Womens E-Mail    0.008837  1.077202


Beyond matching group sizes, a randomized assignment should also leave any pre-treatment customer covariates similarly distributed across the three groups, not just the group counts themselves; two groups can be the same size and still be unbalanced on who is in them. We check every other numeric column in the file this way, whatever it turns out to contain, rather than assuming `segment`, `conversion`, and `spend` are the only columns worth a second look.

In [4]:
covariate_cols = [
    c for c in df.columns
    if c not in ("segment", "conversion", "spend") and pd.api.types.is_numeric_dtype(df[c])
]

if covariate_cols:
    print(f"Numeric covariates found: {covariate_cols}")
    print(df.groupby("segment")[covariate_cols].mean())
else:
    print("No additional numeric covariates in this file beyond conversion and spend.")

Numeric covariates found: ['recency', 'history', 'mens', 'womens', 'newbie', 'visit']
                recency     history      mens    womens    newbie     visit
segment                                                                    
Mens E-Mail    5.773642  242.835931  0.550946  0.551415  0.501525  0.182757
No E-Mail      5.749695  240.882653  0.553224  0.547639  0.501971  0.106167
Womens E-Mail  5.767850  242.536633  0.548932  0.550101  0.503250  0.151400


Groups are well-balanced on size (21,387 Womens, 21,307 Mens, 21,306 No E-Mail) and, per the covariate check above, on every other numeric column available in the file, consistent with a properly randomized experimental design, not something we need to correct for.

The raw numbers already suggest a pattern worth testing formally: Mens
E-Mail shows the highest conversion rate (1.25%, more than double the
control's 0.57%) and highest average spend ($1.42 vs. control's $0.65).
Womens E-Mail sits between the two on both metrics (0.88% conversion,
$1.08 spend). Whether these differences are statistically significant, 
and whether they survive Bonferroni correction, is what Sections 2-3 test
formally, rather than concluding anything from these raw averages alone.

## 2. Conversion: Two-Proportion Z-Tests

Conversion is binary, so each treatment group is compared against control
using a two-proportion z-test, the standard test for comparing rates
between two independent groups.

In [5]:
control = df[df["segment"] == "No E-Mail"]
mens = df[df["segment"] == "Mens E-Mail"]
womens = df[df["segment"] == "Womens E-Mail"]

results_conversion = []

for name, group in [("Mens", mens), ("Womens", womens)]:
    count = np.array([group["conversion"].sum(), control["conversion"].sum()])
    nobs = np.array([len(group), len(control)])
    z_stat, p_value = proportions_ztest(count, nobs)
    results_conversion.append({
        "comparison": f"{name} vs Control",
        "metric": "conversion",
        "group_rate": group["conversion"].mean(),
        "control_rate": control["conversion"].mean(),
        "z_stat": z_stat,
        "p_value": p_value,
    })

results_conversion_df = pd.DataFrame(results_conversion)
results_conversion_df

,comparison,metric,group_rate,control_rate,z_stat,p_value
0,Mens vs Control,conversion,0.012531,0.005726,7.385114,1.523224e-13
1,Womens vs Control,conversion,0.008837,0.005726,3.779561,1.571051e-04


Both campaigns show statistically significant increases in conversion rate,
even before applying the Bonferroni correction. Mens vs. Control: 1.25% vs.
0.57%, z = 7.39, p ≈ 1.5×10⁻¹³. Womens vs. Control: 0.88% vs. 0.57%, z =
3.78, p ≈ 1.6×10⁻⁴. Both p-values are far below the Bonferroni-adjusted
threshold of 0.0125 (0.05/4), so both survive correction comfortably, this
isn't a borderline result that correction might flip.

## 3. Spend: Two-Sample T-Tests

Spend is a continuous dollar amount (zero for non-converters), so each
treatment group is compared against control using an independent two-sample
t-test, Welch's variant, which doesn't assume equal variance between
groups, consistent with the approach used in `05_campaigns_and_coupons.ipynb`. With over 21,000 customers per group, the Central Limit Theorem makes the t-test's normality assumption a non-issue at the level of the *sample mean*, even though `spend` itself is nowhere near normal (a spike at zero for the ~98% who don't convert, a right-skewed tail for those who do); this is the standard justification for using a t-test on revenue-like metrics in large-sample A/B testing, and is worth stating rather than leaving the reader to wonder whether the skew invalidates the test.

In [6]:
results_spend = []

for name, group in [("Mens", mens), ("Womens", womens)]:
    t_stat, p_value = stats.ttest_ind(group["spend"], control["spend"], equal_var=False)
    results_spend.append({
        "comparison": f"{name} vs Control",
        "metric": "spend",
        "group_mean": group["spend"].mean(),
        "control_mean": control["spend"].mean(),
        "t_stat": t_stat,
        "p_value": p_value,
    })

results_spend_df = pd.DataFrame(results_spend)
results_spend_df

,comparison,metric,group_mean,control_mean,t_stat,p_value
0,Mens vs Control,spend,1.422617,0.652789,5.300140,1.163815e-07
1,Womens vs Control,spend,1.077202,0.652789,3.256372,1.129397e-03


Both campaigns show a statistically significant increase in spend even
before correction: Mens vs. Control (t = 5.30, p ≈ 1.16×10⁻⁷) and Womens
vs. Control (t = 3.26, p ≈ 1.13×10⁻³). The pattern matches Section 2
exactly, Mens shows the stronger effect on both metrics tested so far,
conversion and spend alike. Whether all 4 results (2 metrics × 2
comparisons) survive joint correction is tested next.

### 3.1 Decomposing the Spend Effect: More Converters, or Bigger Baskets?

The spend test above answers "does the average dollar amount, across everyone including non-converters, go up," which conflates two different effects: campaigns could lift spend purely by creating more converters (already confirmed in Section 2), by making converters spend more once they convert, or both. We check the second piece directly, restricting the same comparison to customers who converted. This is an exploratory, complementary check, not part of the 4-test family Section 4 corrects for.

In [7]:
mens_converters = mens[mens["conversion"] == 1]
womens_converters = womens[womens["conversion"] == 1]
control_converters = control[control["conversion"] == 1]

for name, group in [("Mens", mens_converters), ("Womens", womens_converters)]:
    t_stat, p_value = stats.ttest_ind(group["spend"], control_converters["spend"], equal_var=False)
    print(f"{name} vs Control, spend among converters only: "
          f"${group['spend'].mean():.2f} vs ${control_converters['spend'].mean():.2f} "
          f"(n={len(group)} vs n={len(control_converters)}), t={t_stat:.3f}, p={p_value:.4f}")

Mens vs Control, spend among converters only: $113.53 vs $114.00 (n=267 vs n=122), t=-0.041, p=0.9672
Womens vs Control, spend among converters only: $121.89 vs $114.00 (n=189 vs n=122), t=0.653, p=0.5145


Whatever this shows, it refines rather than replaces Section 3's result: the overall spend lift there is real either way, this just clarifies whether it comes from more people converting, from converters spending more once they do, or both — a distinction worth having on hand for the business interpretation in Section 5.

## 4. Multiple Comparisons Correction

Section 2 (conversion) and Section 3 (spend) produced 4 p-values in total, 2 comparisons × 2 metrics. Bonferroni correction is applied jointly across all 4, not separately within each metric, since all 4 tests belong to the same family of comparisons defined in the introduction.

Bonferroni divides the significance threshold by the number of tests: 0.05 / 4 = 0.0125. Any p-value above this adjusted threshold is not considered significant after correction, even if it would have been at the uncorrected 0.05 level.

In [8]:
all_results = pd.concat([results_conversion_df, results_spend_df], ignore_index=True)

reject, p_adjusted, _, alpha_bonferroni = multipletests(
    all_results["p_value"], alpha=0.05, method="bonferroni"
)

all_results["p_adjusted"] = p_adjusted
all_results["significant_after_correction"] = reject

print(f"Bonferroni-adjusted alpha: {alpha_bonferroni:.4f}")
all_results

Bonferroni-adjusted alpha: 0.0125


,comparison,metric,group_rate,control_rate,z_stat,p_value,group_mean,control_mean,t_stat,p_adjusted,significant_after_correction
0,Mens vs Control,conversion,0.012531,0.005726,7.385114,1.523224e-13,NaN,NaN,NaN,6.092894e-13,True
1,Womens vs Control,conversion,0.008837,0.005726,3.779561,1.571051e-04,NaN,NaN,NaN,6.284205e-04,True
2,Mens vs Control,spend,NaN,NaN,NaN,1.163815e-07,1.422617,0.652789,5.300140,4.655260e-07,True
3,Womens vs Control,spend,NaN,NaN,NaN,1.129397e-03,1.077202,0.652789,3.256372,4.517588e-03,True


All 4 tests remain significant after Bonferroni correction (adjusted alpha ≈ 0.0125). Neither campaign's effect on conversion or spend was a borderline result that correction could have flipped, every raw p-value was already several orders of magnitude below the adjusted threshold.

(The `NaN` values are expected, concatenating the two results tables means
each row only populates the columns relevant to its own test type.)

## 5. Business Interpretation

Both campaigns produced a statistically significant lift over control, on both conversion and spend, surviving Bonferroni correction comfortably. Mens E-Mail shows the stronger effect on both metrics: conversion more than doubles (1.25% vs. 0.57% control), and mean spend more than doubles as well ($1.42 vs. $0.65).

A caveat worth stating explicitly: with over 21,000 customers per group, even a small absolute difference becomes statistically detectable, statistical significance here confirms the effect is real, not that it is large in absolute terms. The mean spend figures are low in absolute dollars precisely because over 98% of customers in every group did not convert at all; this is an average diluted by a large majority of zeros, not a typical per-converting-customer spend. A business decision to scale the Mens campaign would still need to weigh this lift against the cost of sending the campaign, a comparison this dataset alone does not provide.

Between the two campaigns, Mens E-Mail is the stronger candidate on the evidence available here, but this microcase focused on demonstrating the experimentation methodology (randomized assignment, appropriately matched tests, multiple-comparisons correction) rather than a full campaign ROI analysis.

In [9]:
print(f"Bonferroni-adjusted alpha: {alpha_bonferroni:.4f}")
all_results[["comparison", "metric", "p_value", "p_adjusted", "significant_after_correction"]]

Bonferroni-adjusted alpha: 0.0125


,comparison,metric,p_value,p_adjusted,significant_after_correction
0,Mens vs Control,conversion,1.523224e-13,6.092894e-13,True
1,Womens vs Control,conversion,1.571051e-04,6.284205e-04,True
2,Mens vs Control,spend,1.163815e-07,4.655260e-07,True
3,Womens vs Control,spend,1.129397e-03,4.517588e-03,True


## 6. Project Conclusion

This microcase closes `P2 — retail-consumer-behavior`. Across 7 notebooks,
the project moved from raw, unverified CSVs to a fully validated PostgreSQL
database (`01`), a household-level RFM table built and scored with evidence
at every step (`02`), a data-driven customer segmentation that explicitly
accounted for K-means' sensitivity to outliers (`03`), a cohort-retention
analysis that correctly distinguished a real signal from a window-edge
artifact (`04`), a campaign-effectiveness analysis that used two
comparison designs specifically to catch a selection-bias trap a single
design would have missed (`05`), and two independent microcases applying
funnel analysis (`06`) and rigorous hypothesis testing with multiple-
comparisons correction (`07`) to different datasets entirely.

Every methodological decision throughout, from excluding non-positive
`sales_value`, to choosing IQR over z-scores for skewed data, to using
Bonferroni instead of Benjamini-Hochberg for only 4 tests, was made and
documented for a reason specific to the data and the question at hand,
not reused automatically from a prior project. That was the standard set
at the start of this project, and it holds through to this final notebook.